# Media Authenticity Detection - CNN Model Training (Optimized & Fast)

This notebook covers the training and evaluation of a custom CNN.

**Speed & Stability Optimizations:**
- **Reduced Image Size (128x128):** ~3x faster training than 224x224 with minimal accuracy loss on CPU.
- **Reduced Batch Size (16):** Prevents RAM crashes.
- **Checkpoints:** Saves progress after every epoch.
- **CPU Enforcement:** Disables GPU overhead.

In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import gc

print(f"TensorFlow Version: {tf.__version__}")

# GPU Check & Optimization
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) detected. Enabling mixed precision.")
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
else:
    print("No GPU detected. Enforcing CPU optimizations.")
    tf.config.set_visible_devices([], 'GPU')

# Paths
BASE_DIR = os.path.join("..", "data", "processed", "images")
MODEL_SAVE_PATH = os.path.join("..", "models", "cnn_model_fast.keras")
MODEL_PKL_PATH = os.path.join("..", "models", "cnn_model.pkl")
CHECKPOINT_DIR = os.path.join("..", "models", "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hyperparameters (Adjusted for Speed)
IMAGE_SIZE = (128, 128) # Reduced from 224x224 for 3x speedup on CPU
BATCH_SIZE = 16
EPOCHS = 20

TensorFlow Version: 2.21.0
No GPU detected. Enforcing CPU optimizations.


## 1. Data Loading

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'val'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'test'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

Found 70000 files belonging to 2 classes.
Found 15000 files belonging to 2 classes.
Found 15000 files belonging to 2 classes.


## 2. Performance Optimization

In [3]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

gc.collect()

0

## 3. Simplified CNN Architecture (Faster)

In [4]:
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

D:\Media Validate App\.venv\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,048,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,105,025 (4.22 MB)

 Trainable params: 1,105,025 (4.22 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Training

In [5]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

checkpoint = callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "cnn_fast_epoch_{epoch:02d}.keras"),
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.8206 - loss: 0.4018
Epoch 1: val_loss improved from None to 0.20561, saving model to ..\models\checkpoints\cnn_fast_epoch_01.keras

Epoch 1: finished saving model to ..\models\checkpoints\cnn_fast_epoch_01.keras
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 896s 202ms/step - accuracy: 0.8681 - loss: 0.3188 - val_accuracy: 0.9205 - val_loss: 0.2056
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.9185 - loss: 0.2211
Epoch 2: val_loss improved from 0.20561 to 0.17998, saving model to ..\models\checkpoints\cnn_fast_epoch_02.keras

Epoch 2: finished saving model to ..\models\checkpoints\cnn_fast_epoch_02.keras
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 784s 177ms/step - accuracy: 0.9212 - loss: 0.2125 - val_accuracy: 0.9343 - val_loss: 0.1800
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 0.9323 - loss: 0.1800
Epoch 3: val_loss improved from 0.17998 to 0.16416, saving model to ..\models\checkpoints\cnn_fas

## 5. Evaluation & Saving

In [6]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc:.4f}")

model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

938/938 ━━━━━━━━━━━━━━━━━━━━ 63s 67ms/step - accuracy: 0.9448 - loss: 0.1568

Test Accuracy: 0.9448
Model saved to ..\models\cnn_model_fast.keras


## 6. Saving to .pkl (Optional)

In [7]:
import pickle
try:
    with open(MODEL_PKL_PATH, 'wb') as f:
        pickle.dump(model, f)
    print(f"Model successfully saved to {MODEL_PKL_PATH}")
except Exception as e:
    print(f"Pickle save failed: {e}")

Model successfully saved to ..\models\cnn_model.pkl
